# 03a — NumPy and Pandas data contracts — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/03.md) · [Course map](../PLAN.md) · [Project](../../projects/stage03/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Inspect shapes and broadcasting
- Clean, join, and summarize tabular data
- Preserve provenance and missingness


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Arrays and tables without shape surprises

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A NumPy array stores values in a regular shape. Shape `(3,)` is a length-three vector; `(3,1)` is three rows with one column. An axis is one dimension; a reduction such as sum can collapse a chosen axis. Broadcasting lets compatible shapes expand logically, but can create unintended matrices if you mix row and column shapes.

A Pandas DataFrame is a table with named columns and an index. Each column has a data type. Missing values, duplicate records, and invalid values are different problems; count and explain them separately. Parse dates before chronological operations.

A join combines rows using keys. A many-to-one join expects the lookup side's key to be unique. If both sides repeat the key, row multiplication can inflate totals. A rolling statistic summarizes a moving window whose order and missing-date policy must be stated.

### Worked example: follow the values

Daily spending is 9 on Monday and 3 on Wednesday. Reindex to all calendar days and use zero for Tuesday under a stated no-recorded-spend convention. Wednesday's three-day mean is `(9+0+3)/3=4`. A three-row window on the original transaction table would answer a different question.

### Pause and explain

Should every large amount flagged as an outlier be deleted?

<details><summary>Check your reasoning after trying</summary>

No. A rare value can be legitimate. Flag it, inspect source/context, and document any exclusion.

</details>

### Common mistakes to check

Check shape before arithmetic, key uniqueness before joining, and whether missing means zero or unknown before filling it.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

NumPy arrays hold regular typed data. Broadcasting aligns axes from the right: dimensions must match or one must equal 1. A (n,1) column minus a (d,) row produces (n,d); accidentally subtracting (n,) from (n,1) produces (n,n). Inspect shape before trusting a result.

Pandas labels rows and columns. Its joins align keys, not row positions. Duplicate keys on both sides can multiply rows, so use merge validation. Missing values represent absent information; zero is an observed value and should not be substituted automatically. Parse dates explicitly, sort before rolling calculations, and document duplicate policy.

A data contract defines fields, types, units, valid ranges, uniqueness, and freshness. Keep rejected records with a reason. Fit statistical imputers only on training data; cleaning based on fixed domain rules is different from learning a transformation from observations.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import numpy as np
import pandas as pd
x = np.array([[1., 2.], [3., 4.]])
print(x.shape, x.mean(axis=0))
print(pd.read_csv(ROOT / "datasets" / "expenses.csv").head())


## Exercise 1: Column standardization

Standardize each column using population std. Constant columns become zeros. Return a NumPy array.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def standardize(x):
    raise NotImplementedError


In [ ]:
z = standardize([[1, 7], [3, 7]])
assert np.allclose(z, [[-1, 0], [1, 0]])
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Replace zero denominators, not output NaNs.

</details>


## Exercise 2: Monthly spending

Parse dates and amounts, group by calendar month string YYYY-MM, return a sorted Series. Use a copied frame.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def monthly_spending(frame):
    raise NotImplementedError


In [ ]:
frame = pd.DataFrame({"date": ["2026-02-01", "2026-01-01", "2026-01-03"], "amount": [4, 2, 3]})
assert monthly_spending(frame).to_dict() == {"2026-01": 5, "2026-02": 4}
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

dt.strftime creates the monthly grouping key.

</details>


## Exercise 3: Safe join

Left-join orders and customers on customer_id; enforce many-to-one and preserve row count.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def enrich(orders, customers):
    raise NotImplementedError


In [ ]:
orders = pd.DataFrame({"customer_id": [1, 1, 2], "amount": [1, 2, 3]})
customers = pd.DataFrame({"customer_id": [1, 2], "name": ["A", "B"]})
assert len(enrich(orders, customers)) == 3
expect_error(pd.errors.MergeError, lambda: enrich(orders, pd.concat([customers, customers])))
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Use the validate parameter.

</details>


## Independent transfer

Profile missing values, duplicates, date ranges, and outliers in expenses.csv. Write a cleaning report and a three-day rolling mean. Explain every exclusion.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [03a interface and acceptance cases](../assignments/03a.md).
2. Copy the [blank starter](../assignments/starters/03a.py) to `work/assignments/03a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 03a --solution work/assignments/03a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/03a.md#03a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 03a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. What is broadcasting?

2. Why validate a join?


## Reading and review

- [Primary reference 1](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [Primary reference 2](https://pandas.pydata.org/docs/user_guide/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
